# Start of the Sensitivity Analysis of the RCMs 

Program to read in the zarr collection of an RCM and its corresponding FFDI files and explore FFDI drivers


3-23-.10




#### required packages

In [ ]:
import intake
import xarray as xr
from matplotlib import pyplot as plt
import numpy as np

from glob import glob
import pathlib
import traceback
from datetime import datetime

from xclim.indices import (
    keetch_byram_drought_index,
    griffiths_drought_factor,
    mcarthur_forest_fire_danger_index
)

%load_ext autoreload
%autoreload 2


# importing sys
import sys
# adding plotting module to the system path
sys.path.insert(0, '/g/data/xv83/rxm599/acs/plotting_maps')
# import ACS plotting maps and Xarray.
from acs_plotting_maps import *
from acs_area_statistics import acs_regional_stats, get_regions

import geopandas as gpd
import pandas as pd
import regionmask

sys.path.insert(0, '/g/data/xv83/rxm599/acs/hazard_fire/paper2')
import sa as sa
import df as df


#### start a local Dask client

In [ ]:
from dask.distributed import Client, LocalCluster
import dask
import os

# --- Dask optimiser-style settings ---
dask.config.set({
    'distributed.comm.timeouts.connect': '90s',  # Timeout for connecting to a worker
    'distributed.comm.timeouts.tcp': '90s',  # Timeout for TCP communications
#    "distributed.worker.memory.target": False,
#    "distributed.worker.memory.spill": False,
#    "distributed.worker.memory.pause": False,
#    "distributed.worker.memory.terminate": False,
})

# --- Match ARE allocation ---
ncpus = int(os.environ.get("PBS_NCPUS", 1))

cluster = LocalCluster(
    n_workers=ncpus,      # one worker per CPU
    threads_per_worker=1, # critical (matches optimiser)
    processes=True,
    memory_limit=0        # removes worker memory limit
)

client = Client(cluster)
client

from dask.distributed import Client
import dask

# Set configuration options
dask.config.set({
    'distributed.comm.timeouts.connect': '90s',  # Timeout for connecting to a worker
    'distributed.comm.timeouts.tcp': '90s',  # Timeout for TCP communications
})

#cluster = LocalCluster(
#    n_workers=28,          # Number of workers
#    threads_per_worker=1 #Threads per worker
#    #memory_limit='8GB' # Memory limit per each worker commented out
#)
#client = Client(cluster)

client = Client()
client

In [ ]:
import warnings
warnings.filterwarnings('ignore')

## Set parameters

In [ ]:
# parameters
# 0,2,3 failed
mindex=0
#lon1=140; lon2=150 
#lat1=-45; lat2= -32
#lat1=-45 lat2= -40 lon1=144 lon2=149
#lon1=145 ; lon2=146
#lat1=-37 ; lat2=-36
lat1=-45
lat2= -10
lon1=110
lon2=155
p_ext=0.95
p_ext=0.99

t1='2015-01-01'
t2='2035-01-01'
dirstore='sa_2020'

In [ ]:
# extra info
mindexp100=mindex+100
mobs=39

## Obtain the desired catalogues of the simulations to processe

In [ ]:
dtype='/g/data/ia39/ncra/fire/bias-output/'
zarr_path=dtype+ 'zarr/*ssp370*'
ffdi_path=dtype+'ffdi/*ssp370*FFDI*'
kbdi_path=dtype+'ffdi/*ssp370*KBDI*'
# observations 
zobs='/g/data/ia39/ncra/fire/bias-input/zarr/*ERA*' 
fobs='/g/data/ia39/ncra/fire/bias-input/ffdi/*ERA*FFDI*'
kobs='/g/data/ia39/ncra/fire/bias-input/ffdi/*ERA*KBDI*'

mRuns = sorted(glob(zarr_path)) + sorted(glob(zobs))
mFFDI = sorted(glob(ffdi_path))+ sorted(glob(fobs))
mKBDI = sorted(glob(kbdi_path)) + sorted(glob(kobs))
print(len(mRuns))
print(len(mFFDI))

# print code to ensure the files match
ifile=-1
for file in mRuns: 
    ifile=ifile+1
#    print(ifile, file)
ifile=-1
for file in mFFDI: 
    ifile=ifile+1
#    print(ifile, file)

# Process one ensemble 

In [ ]:
# From one catalogue list save variables
ds0=xr.open_zarr(mRuns[mindex])
df0=xr.open_zarr(mFFDI[mindex])
dk0=xr.open_zarr(mKBDI[mindex])

print(mRuns[mindex])
print(mFFDI[mindex])
print(mKBDI[mindex])


# compute DF 
# kbdi has a 20 extra points at the start for DF calculation
if mindex != mobs: 
# for projections    
    pra = ds0.prAdjust.sel(time=slice(ds0.prAdjust.time[0],'2099-12-31'))
    kbdi=dk0.KBDI.sel(time=slice(pra.time[0],pra.time[-1] ))
    DF = df.griffiths_drought_factor_dask_exact(pra, kbdi)
else:
    print("obs")
    pra = ds0.pr.sel(time=slice(ds0.pr.time[0],'2099-12-31'))
    kbdi=dk0.KBDI.sel(time=slice(pra.time[0],pra.time[-1] ))
    DF = df.griffiths_drought_factor_dask_exact(pra, kbdi)
    
print(pra)
print(kbdi)


In [ ]:
# compute the 95% value from first 20 years
if mindex != mobs:
    tr1='2015-01-01'; tr2='2035-01-01'
else:
    tr1='2000-01-01'; tr2='2020-01-01'
    t1=tr1; t2=tr2
#    tr1='2003-01-01'; tr2='2023-12-31'

df1=df0.sel(time=slice(tr1,tr2))
d95a=df1.FFDI.quantile(p_ext,dim='time')


## mask info

In [ ]:
%%time
# NCRA regions from acs_area_statistics code
# these are the names of your regions
regions = get_regions([
                           "australia"
                      ])
regions
# nrm_regions ncra_regions",


In [ ]:
mask_frac = regions.mask_3D(d95a)
#mask_frac = regions.mask_3D_frac_approx(d95) # not defined in 3-23.10

In [ ]:
mask_ = mask_frac.isel(region=0)
d95=d95a.where(mask_).load()
d95.plot(cmap='plasma',levels=10)

In [ ]:
df2=df0.sel(time=slice(t1,t2))
ds2=ds0.sel(time=slice(t1, t2))
dk2=dk0.sel(time=slice(t1,t2))
dd2=DF.sel(time=slice(t1,t2))
# extract only the values greater than FFDI > 95%
dfe95=df0.FFDI.where(df2.FFDI > d95)
dse95=ds0.where(df2.FFDI > d95)
dke95=dk0.where(df2.FFDI > d95)
dde95=DF.where(df2.FFDI > d95)

In [ ]:
# fixed naming in UQ-DEC files
print(list(dse95.data_vars))
var=list(dse95.data_vars)
for jj in list(dse95.data_vars):
    if (jj == 'sfcWindAdjust'):
        qt=True
        ddnew=dse95.rename({'sfcWindAdjust': 'sfcWindmaxAdjust'})
        dse95=ddnew

print(list(dse95.data_vars))

In [ ]:
dde95.load()

In [ ]:
%%time
# Reduce workers for IO phase
cluster.scale(4)   # e.g. 2–6 workers is usually good

# Only create file if it does not exist
outf='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_dde95.zarr'
print(outf)
if os.path.exists(outf):
    print("File exists!")
    ddtmp=xr.open_zarr(outf)
else:
    print("File does not exist.")
    dde95.to_zarr(outf, mode='w',zarr_format=2)
    ddtmp=xr.open_zarr(outf)

cluster.scale(n_workers_full)

In [ ]:
dde95

In [ ]:
#print(dde95)
var_name = list(ddtmp.data_vars)[0]
dde95 = ddtmp[var_name]
#dde95

In [ ]:

if mindex != mobs: 
# for projections    
    tmax=dse95.tasmaxAdjust.sel(lon=slice(lon1,lon2),lat=slice(lat1,lat2 )) 
    hmin=dse95.hursminAdjust.sel( lon=slice(lon1,lon2),lat=slice(lat1,lat2 )) 
    wmax=dse95.sfcWindmaxAdjust.sel(lon=slice(lon1,lon2),lat=slice(lat1,lat2 )) 
else:
    tmax=dse95.tasmax.sel(lon=slice(lon1,lon2),lat=slice(lat1,lat2 )) 
    hmin=dse95.hursmin.sel( lon=slice(lon1,lon2),lat=slice(lat1,lat2 )) 
    wmax=dse95.sfcWindmax.sel(lon=slice(lon1,lon2),lat=slice(lat1,lat2 )) 
    
# ffdi 
di=dde95.sel(lon=slice(lon1,lon2),lat=slice(lat1,lat2 )) 
b=dfe95.sel(lon=slice(lon1,lon2),lat=slice(lat1,lat2 )) 
#di.max(axis=2).plot(cmap='plasma')
#di

## Fit PDF to use in Sobol calculation

In [ ]:
# generic definitions of fitting function

# calculate min and remove it before fitting
ta=tmax.min(dim='time')
ha=hmin.min(dim='time')
da=di.min(dim='time')
wa=wmax.min(dim='time')

tmax1=tmax-ta
hmin1=hmin-ha
di1=di-da
wmax1=wmax-wa

tfit=sa.fit_gamma(tmax1)
hfit=sa.fit_gamma(hmin1)
dfit=sa.fit_gamma(di1)
wfit=sa.fit_gamma(wmax1)

tfit1=sa.fit_lognorm(tmax1)
hfit1=sa.fit_lognorm(hmin1)
dfit1=sa.fit_lognorm(di1)
wfit1=sa.fit_lognorm(wmax1)


%%time
tfit=sa.fit_beta(tmax).load()
hfit=sa.fit_beta(hmin).load()
dfit=sa.fit_gamma(di).load()
wfit=sa.fit_beta(wmax).load()

dfita=sa.fit_beta(di).load()
dfitb=sa.fit_lognorm(di).load()

In [ ]:
%%time
outf1='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_fits.zarr'
outf2='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_lfits.zarr'
print(mindex)
print(outf1)
tfit.name='tfit'
hfit.name='hfit'
wfit.name='wfit'
dfit.name='dfit'

tfit1.name='tfit'
hfit1.name='hfit'
wfit1.name='wfit'
dfit1.name='dfit'

ta.name='Tmax_min'
ha.name='Hmin_min'
da.name='DI_min'
wa.name='Wmax_min'

la=-30;lo=115.05
print(tfit.sel(lon=lo,lat=la).values)
print(tfit1.sel(lon=lo,lat=la).values)

#ds= xr.merge([ta,ha,da,wa])
ds = xr.merge([tfit,hfit,wfit,dfit,ta,ha,da,wa])  #,dfit,dfitb])
ds.to_zarr(outf1,mode='w',zarr_format=2)

ta.name='Tmax_min'
ha.name='Hmin_min'
da.name='DI_min'
wa.name='Wmax_min'

#ds = xr.merge([tfit,hfit,wfit,dfita,dfit,dfitb])
#ds= xr.merge([ta,ha,da,wa])
dslog = xr.merge([tfit1,hfit1,wfit1,dfit1])  #,dfit,dfitb])

dslog.to_zarr(outf2,mode='w',zarr_format=2)

## Read in fit results

In [ ]:
%%time
ds1=0; ds2=0
outf1='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_fits.zarr'
print(mindex)
print(outf1)

ds1 = xr.open_zarr(outf1,zarr_format=2)

In [ ]:
%%time
tfit=ds1.tfit.load()
hfit=ds1.hfit.load()
wfit=ds1.wfit.load()
dfit=ds1.dfit.load()

In [ ]:
lev1=np.arange(0, 1, .1)
lev1 = np.linspace(0, .0001, 11)
plt.figure(figsize=(8, 8))
plt.subplot(2,2,1); tfit.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Tmax")
plt.subplot(2,2,2); hfit.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Hmin")
plt.subplot(2,2,3); dfit.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("DI")
plt.subplot(2,2,4); wfit.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Wmax")

In [ ]:
%%time

def bad(a11): # Find locations where condition is True
    locations = a11 < .0000002
# Stack dimensions into a flat index
    stacked = locations.stack(all_points=("lon", "lat"))
# Get the coordinates where the condition is True
    matching_indices = stacked.where(stacked, drop=True)
# Print the matching indices
#    print(matching_indices.coords)
    count = (matching_indices).sum().item()  # .item() to get Python scalar
    print('number of points=',count)
    return matching_indices

res=bad(tfit.sel(parameter1='p-value'))
res=bad(hfit.sel(parameter1='p-value'))
res=bad(dfit.sel(parameter1='p-value'))
res=bad(wfit.sel(parameter1='p-value'))


### Do it again with lognorm fits

In [ ]:
%%time
outf2='/scratch/xv83/rxm599/'+f'{dirstore}/'+f'model_{mindexp100}_lfits.zarr'
print(mindex)
print(outf1)

ds2 = xr.open_zarr(outf2,zarr_format=2)

In [ ]:
%%time
tfit1=ds2.tfit.load()
hfit1=ds2.hfit.load()
wfit1=ds2.wfit.load()
dfit1=ds2.dfit.load()

In [ ]:
lev1=np.arange(0, 1, .1)
lev1 = np.linspace(0, .0001, 11)
plt.figure(figsize=(8, 8))
plt.subplot(2,2,1); tfit1.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Tmax")
plt.subplot(2,2,2); hfit1.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Hmin")
plt.subplot(2,2,3); dfit1.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("DI")
plt.subplot(2,2,4); wfit1.sel(parameter1='p-value').plot(levels=lev1,cmap='plasma')
plt.title("Wmax")

In [ ]:
%%time

res=bad(tfit1.sel(parameter1='p-value'))
res=bad(hfit1.sel(parameter1='p-value'))
res=bad(dfit1.sel(parameter1='p-value'))
res=bad(wfit1.sel(parameter1='p-value'))


In [ ]:
a1=(tfit.sel(parameter1='p-value') - tfit1.sel(parameter1='p-value'))
#a1.max().values
la=-30;lo=115.05
print(tfit.sel(lon=lo,lat=la).values)
print(tfit1.sel(lon=lo,lat=la).values)

In [ ]:
a1=wfit.sel(parameter1='p-value')
a2=wfit1.sel(parameter1='p-value')
b=xr.apply_ufunc(np.maximum, a1, a2)
res=bad(b)
a1=tfit.sel(parameter1='p-value')
a2=tfit1.sel(parameter1='p-value')
b=xr.apply_ufunc(np.maximum, a1, a2)
res=bad(b)
a1=hfit.sel(parameter1='p-value')
a2=hfit1.sel(parameter1='p-value')
b=xr.apply_ufunc(np.maximum, a1, a2)
res=bad(b)
a1=dfit.sel(parameter1='p-value')
a2=dfit1.sel(parameter1='p-value')
b=xr.apply_ufunc(np.maximum, a1, a2)
res=bad(b)

In [ ]:
sys.exit(1)

## Testing fit to a location 

In [ ]:
la=-30;lo=115.05
#tmp1=di1.sel(lat=la,lon=lo).load()
tmp1=tmax1.sel(lat=la,lon=lo).load()
#tmp1=wmax1.sel(lat=la,lon=lo).load()
#tmp1=hmin1.sel(lat=la,lon=lo).load()
dfita=sa.fit_gamma(tmp1)
dfitb=sa.fit_invgauss(tmp1)
dfitc=sa.fit_lognorm(tmp1)

dfit1a=dfita.load()
dfit1b=dfitb.load()
dfit1c=dfitc.load()
#print(dfit1a)
#dfit1=dfit.sel(lat=la,lon=lo).load()
#dfit1a=dfita.sel(lat=la,lon=lo).load()
#dfit1b=dfitb.sel(lat=la,lon=lo).load()

In [ ]:
%%time
# Testing the fitting at one location by comparing beta and gamma fits
from scipy.stats import beta, kstest, gamma, lognorm, invgauss
#tmp1=tmax.sel(lat=la,lon=lo).load()
#tmp1=tmp1-tmp1.min()

print(dfit1a)
print(dfit1b)
print(dfit1c)
tmp1.plot.hist(density=True)

rvb=invgauss(dfit1b.sel(parameter1='a'),
        dfit1b.sel(parameter1='loc'),dfit1b.sel(parameter1='scale') )
rvg=gamma(dfit1a.sel(parameter1='a'),
        dfit1a.sel(parameter1='loc'),dfit1a.sel(parameter1='scale') )
rvl=lognorm(dfit1c.sel(parameter1='a'),
        dfit1c.sel(parameter1='loc'),dfit1c.sel(parameter1='scale') )
plt.plot(tmp1,rvg.pdf(tmp1),'o',label='gamma')
plt.plot(tmp1,rvb.pdf(tmp1),'go',label='invgauss')
plt.plot(tmp1,rvl.pdf(tmp1),'ro',label='lognorm')
plt.legend()


In [ ]:
#client.shutdown()